# Proyecto ML - parte 1 — Envío 5

## Roles de opinión + negaciones + RidgeClassifier

El objetivo de este envío es mejorar la clasificación de reseñas en español mediante una representación más precisa del lenguaje negado y un clasificador final `RidgeClassifier`.

La explicación detallada de los cambios con respecto al envío 4 se presenta en la sección comparativa siguiente. Los resultados de la evaluación se reservan para la sección final.

## 1. Datos y módulo de características

El notebook carga el módulo local `features_copia.py`, donde se encuentra la implementación del analizador de reseñas. Separar esta lógica permite mantener el notebook organizado y reutilizar el mismo transformador durante la validación cruzada, el entrenamiento final y la predicción.

El módulo se mantiene independiente de `features_grupo18.py` para que los experimentos del envío 5 no modifiquen el código ni los resultados del envío 4. Las funciones que conserva y las características nuevas que incorpora se explican en la sección comparativa siguiente.

## Diferencias específicas con respecto al envío 4

Es importante anotar que el envío 5 no reemplaza la estrategia del envío 4; la toma como punto de partida y añade una mejora enfocada en la interpretación del lenguaje negado.

### Elementos que se mantienen

Los dos envíos utilizan la misma idea central: una reseña puede contener varias cláusulas y no todas tienen la misma importancia para determinar la etiqueta. Por eso se conservan:

- la separación del texto en cláusulas mediante puntuación y conectores discursivos;
- la detección aprendida de cláusulas de opinión frente a cláusulas logísticas o de relleno;
- la identificación de la opinión más reciente y de las opiniones anteriores;
- los roles `O1`, `O2`, `O3` y `F`, donde `O1` representa la opinión más reciente;
- los marcadores discursivos, como «pero», «sin embargo», «al final» y «en resumen»;
- las vistas `epl`, `roles`, `ultima` y `o1` que luego se combinan mediante `ColumnTransformer`;
- la validación sin fuga de información, porque el analizador y los vectorizadores se ajustan dentro de cada fold.

### Cambios introducidos en el envío 5

1. **Tratamiento explícito de negaciones.** En el envío 4, una palabra como «bueno» se representaba mediante su forma normalizada, pero no se diferenciaba directamente si aparecía afirmada o bajo una negación. El envío 5 activa `usar_negaciones=True` y transforma las palabras afectadas por `no`, `nunca`, `jamás`, `sin`, `ni` o `tampoco` en tokens con prefijo `NOT_`. Por ejemplo:

   ```text
   no es bueno
   ```

   puede aportar tokens equivalentes a:

   ```text
   no NOT_buen
   ```

   Esto evita que «es bueno» y «no es bueno» queden representados casi de la misma manera.

2. **Alcance local de la negación.** La marca `NOT_` no se propaga indefinidamente por toda la reseña. Se reinicia al encontrar puntuación o al comenzar otra cláusula. Así, en una frase como «no es bueno, pero funciona», la negación afecta a la primera valoración y no convierte automáticamente en negativa la segunda.

3. **Clasificador final simplificado.** El envío 4 utilizaba `LinearSVC`. Durante el desarrollo del envío 5 se compararon `LinearSVC` y `RidgeClassifier` utilizando la misma representación con negaciones y las mismas particiones de validación cruzada. `RidgeClassifier` obtuvo un accuracy promedio ligeramente superior, por lo que se eligió como modelo final. El cambio no se hizo por reemplazar automáticamente el clasificador anterior, sino porque la evaluación experimental mostró que Ridge aprovechaba mejor las características TF-IDF, los roles de opinión y los tokens `NOT_` en este conjunto de datos. Por ello, el notebook final utiliza directamente `RidgeClassifier(alpha=1.0)` y no vuelve a entrenar `LinearSVC`.

4. **Módulo de características independiente.** El envío 5 usa `features_copia.py`, una copia de la lógica del envío 4 ampliada con el tratamiento de negaciones. Esto permite experimentar sin modificar `features_grupo18.py` ni alterar los resultados reproducibles del envío 4.



In [8]:
import os
import sys

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

sys.path.insert(0, os.getcwd())
import features_copia as F

df_train = pd.read_csv('train.csv')
y_train = df_train['label']
print(f'{len(df_train)} reseñas | clases: {y_train.value_counts().to_dict()}')

12000 reseñas | clases: {'positivo': 4212, 'negativo': 4212, 'neutral': 3576}


## 2. Representación con negaciones

El analizador conserva las cinco vistas informativas del envío 4. Cada vista ofrece una perspectiva distinta de la reseña: la posición de las cláusulas, el rol discursivo, la última opinión relevante y las opiniones más importantes para la decisión final.

La diferencia principal es `usar_negaciones=True`. El módulo reconoce `no`, `nunca`, `jamás`, `sin`, `ni` y `tampoco`. Desde ese punto, las palabras relevantes reciben el prefijo `NOT_` hasta encontrar un signo de puntuación o terminar la cláusula. El stemming se aplica antes de formar el token, por lo que «bueno» puede producir `NOT_buen`.

Este alcance es local: una negación no debe modificar cláusulas posteriores que expresan una idea independiente. De esta forma se evita confundir una reseña como «no es bueno, pero funciona» con una reseña completamente negativa.

Además de las palabras completas, se usan n-gramas de palabras y de caracteres. Los n-gramas de palabras capturan combinaciones como `no funcion`, mientras que los de caracteres ayudan a reconocer variantes morfológicas, errores pequeños y fragmentos compartidos entre palabras.

In [9]:
def tfidf_marcado(**kwargs):
    params = {
        'ngram_range': (1, 2),
        'sublinear_tf': True,
        'min_df': 2,
        'token_pattern': r'\S+',
        'lowercase': False,
    }
    params.update(kwargs)
    return TfidfVectorizer(**params)


def tfidf_caracteres():
    return TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 5),
        min_df=3,
        sublinear_tf=True,
    )


representacion = ColumnTransformer([
    ('epl', tfidf_marcado(), 'epl'),
    ('roles', tfidf_marcado(), 'roles'),
    ('car_ult', tfidf_caracteres(), 'ultima'),
    ('car_o1', tfidf_caracteres(), 'o1'),
    ('pal_o1', TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=2), 'o1'),
])


def construir_modelo(clasificador):
    return Pipeline([
        ('analizador', F.AnalizadorDeResenas(usar_negaciones=True)),
        ('representacion', representacion),
        ('clf', clasificador),
    ])


modelo_ridge = construir_modelo(RidgeClassifier(alpha=1.0))

In [10]:
ejemplo = df_train['text'].iloc[0]
print('Texto original:\n', ejemplo)
print('\nTokens con negación dentro de la cláusula:\n', F.preprocesar_clausula(ejemplo, usar_negaciones=True))

Texto original:
 Lo pedí un martes por la noche y me llegó al día siguiente. Es el modelo en color negro, el básico de la línea. Lo pesé en la báscula de la cocina y marca algo más de medio kilo. Lo tengo en el escritorio del trabajo y por ahora nomás lo uso de vez en cuando. Viene con su cable y el manual en la caja.

Tokens con negación dentro de la cláusula:
 ['pedi', 'mart', 'noch', 'lleg', 'dia', 'siguient', 'model', 'color', 'negr', 'basic', 'line', 'pes', 'bascul', 'cocin', 'marc', 'mas', 'medi', 'kil', 'escritori', 'trabaj', 'ahor', 'nom', 'uso', 'vez', 'vien', 'cabl', 'manual', 'caj']


## 3. Validación cruzada

Se evalúa exclusivamente `RidgeClassifier` con validación cruzada estratificada de cinco particiones (`5-fold`). En cada iteración, cuatro particiones se utilizan para entrenar y una para validar; este proceso se repite hasta que cada partición haya sido usada como validación.

La estratificación mantiene una proporción similar de las clases `positivo`, `negativo` y `neutral` en todos los folds. `shuffle=True` mezcla las filas antes de dividirlas y `random_state=42` garantiza que el experimento sea reproducible.

El pipeline completo se ajusta dentro de cada fold. Esto incluye el detector de opiniones, los vectorizadores TF-IDF y el clasificador. Por ello, la validación evita fuga de información: la reseña de validación no influye en el vocabulario ni en los parámetros aprendidos durante ese fold.

El argumento `n_jobs=-1` permite ejecutar los folds en paralelo usando los núcleos disponibles, sin cambiar el resultado estadístico.

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_ridge = cross_val_score(modelo_ridge, df_train['text'], y_train, cv=cv, scoring='accuracy', n_jobs=-1)

print('RidgeClassifier por fold:', scores_ridge.round(4))
print(f'RidgeClassifier promedio: {scores_ridge.mean():.4f} (+/- {scores_ridge.std():.4f})')
print('Envío 4 como referencia: 0.9052')

RidgeClassifier por fold: [0.9062 0.9004 0.91   0.9117 0.9021]
RidgeClassifier promedio: 0.9061 (+/- 0.0044)
Envío 4 como referencia: 0.9052


In [12]:
resultados = pd.DataFrame({
    'modelo': ['Envío 4 - LinearSVC sin NOT_', 'Envío 5 - RidgeClassifier con NOT_'],
    'accuracy_cv': [0.9052, scores_ridge.mean()],
})
resultados['mejora_vs_envio_4'] = resultados['accuracy_cv'] - 0.9052
print(resultados.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

                            modelo  accuracy_cv  mejora_vs_envio_4
      Envío 4 - LinearSVC sin NOT_       0.9052             0.0000
Envío 5 - RidgeClassifier con NOT_       0.9061             0.0009


## 4. Diagnóstico opcional del mejor modelo

Después de la validación se fija directamente `modelo_ridge` como el modelo final, porque ya no se realiza una comparación con `LinearSVC`. El reporte detallado por clase se conserva como una opción de diagnóstico.

`cross_val_predict` requiere volver a entrenar el pipeline en cada fold para producir predicciones fuera de muestra. Por eso `GENERAR_REPORTE_CV` está en `False` durante la ejecución normal: así se evita repetir cinco entrenamientos costosos. Si se necesita revisar precisión, recall y F1 por clase, se puede cambiar a `True` y ejecutar nuevamente la celda para visualizar los resultados.

In [13]:
mejor_modelo = modelo_ridge
nombre_mejor = 'ridge'
GENERAR_REPORTE_CV = False

print(f'Mejor modelo nuevo: {nombre_mejor}')
if GENERAR_REPORTE_CV:
    from sklearn.model_selection import cross_val_predict
    y_pred = cross_val_predict(mejor_modelo, df_train['text'], y_train, cv=cv, n_jobs=-1)
    print(classification_report(y_train, y_pred, digits=4))
else:
    print('Reporte CV omitido para evitar cinco entrenamientos adicionales.')

Mejor modelo nuevo: ridge
Reporte CV omitido para evitar cinco entrenamientos adicionales.


## 5. Entrenamiento final y archivo de Kaggle

Una vez validado el enfoque, `RidgeClassifier` se entrena nuevamente utilizando el 100 % de `train.csv`. Este modelo final aprovecha todas las reseñas disponibles y no se reserva una partición interna para validación.

Después se cargan `eval.csv` y `sample_submission.csv`. El pipeline transforma las reseñas de evaluación exactamente con las mismas reglas aprendidas durante el entrenamiento y genera una etiqueta para cada fila. Finalmente, se guarda un CSV con la columna `answer`, respetando el formato solicitado.

También se serializa el pipeline completo con `joblib`. Esto permite reutilizar el modelo sin reconstruir manualmente el analizador, los vocabularios TF-IDF y el clasificador.

In [14]:
mejor_modelo.fit(df_train['text'], y_train)

df_eval = pd.read_csv('eval.csv')
df_sub = pd.read_csv('sample_submission.csv')
df_sub['answer'] = mejor_modelo.predict(df_eval['text'])

archivo_salida = f'submission_{nombre_mejor}_negacion_v5_grupo18.csv'
df_sub.to_csv(archivo_salida, index=False)
joblib.dump(mejor_modelo, f'modelo_{nombre_mejor}_negacion_v5_grupo18.joblib')

print(f'Archivo generado: {archivo_salida}')
print('Distribución de predicciones:')
print(df_sub['answer'].value_counts(normalize=True).round(4))

Archivo generado: submission_ridge_negacion_v5_grupo18.csv
Distribución de predicciones:
answer
positivo    0.367
negativo    0.357
neutral     0.276
Name: proportion, dtype: float64


## 6. Resultados finales

La validación cruzada de cinco folds produjo un accuracy promedio de **90,61 %** para el envío 5. El envío 4 obtuvo **90,52 %** bajo su configuración original.

| Versión | Representación | Clasificador | Accuracy promedio |
|---|---|---|---:|
| Envío 4 | Roles, cláusulas y marcadores, sin tokens `NOT_` | `LinearSVC` | 90,52 % |
| Envío 5 | Representación del envío 4 más tratamiento de negaciones | `RidgeClassifier` | **90,61 %** |

La mejora fue de **0,09 puntos porcentuales**, equivalente a aproximadamente 11 aciertos adicionales sobre las 12.000 reseñas de entrenamiento. Aunque el aumento numérico es pequeño, el envío 5 representa de forma más explícita el cambio de polaridad producido por una negación.

La distribución de las predicciones generadas para el conjunto de evaluación fue:

- positivo: 36,7 %;
- negativo: 35,7 %;
- neutral: 27,6 %.

El archivo de submission generado es `submission_ridge_negacion_v5_grupo18.csv` y el pipeline entrenado queda guardado en `modelo_ridge_negacion_v5_grupo18.joblib`.